# Phase 1 Final Controls Metric Formula Decision

Claim-closed orchestration notebook. This notebook records the manual metric-formula decision after the revision-plan artifact. It does not edit logits, metrics or thresholds, does not rerun controls, and does not open Phase 1 claims.

Scientific-integrity rule: choose a decision because it is justified by the locked contract or documented revision policy, not because it improves an observed result.

In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import hashlib
import json
import subprocess
from datetime import datetime, timezone

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    printable = ' '.join(str(part) for part in cmd)
    if cmd and cmd[0] == 'git':
        printable = printable.replace(str(REPO_URL), 'https://github.com/BrianNguyen29/eeg-ds004752.git')
    print('$', printable)
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo exists:', REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR, check=False)

if RUN_UNITTESTS:
    run(['python', '-m', 'unittest', 'tests.unit.test_phase1_final_controls_metric_formula_decision'], cwd=REPO_DIR)


In [ ]:
# Cell 2 - Select reviewed source artifact and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Use None to resolve latest.txt, or pin an already reviewed formula revision-plan run.
PINNED_FORMULA_REVISION_PLAN_RUN = None

# Manual decision. Keep None until the decision memo is reviewed.
# Allowed values after review: 'raw_ba_ratio', 'gain_over_chance_ratio', or 'unresolved'.
FORMULA_DECISION = None
DECISION_RATIONALE = ''

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_metric_formula_decision'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_metric_formula_decision_manual_hold'

RUN_FINAL_CONTROLS_METRIC_FORMULA_DECISION = False
REQUIRED_ACK = 'I reviewed the metric formula decision and understand it changes no thresholds metrics logits reruns or claims'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_controls_metric_formula_decision_v1_20260423'
print('Notebook fix marker:', FIX_MARKER)


def resolve_latest(root: Path, required_file: str) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        candidate = Path(latest.read_text().strip())
        if (candidate / required_file).exists():
            return candidate
    runs = sorted(path for path in root.iterdir() if path.is_dir()) if root.exists() else []
    for candidate in reversed(runs):
        if (candidate / required_file).exists():
            return candidate
    raise FileNotFoundError(f'No run containing {required_file} under {root}')

FORMULA_REVISION_PLAN_RUN = Path(PINNED_FORMULA_REVISION_PLAN_RUN) if PINNED_FORMULA_REVISION_PLAN_RUN else resolve_latest(
    ARTIFACT_ROOT / 'phase1_final_controls_metric_formula_revision_plan',
    'phase1_final_controls_metric_formula_revision_plan_summary.json',
)

print('Formula revision-plan run:', FORMULA_REVISION_PLAN_RUN)
print('Formula decision:', FORMULA_DECISION)
print('Run flag:', RUN_FINAL_CONTROLS_METRIC_FORMULA_DECISION)


In [ ]:
# Cell 3 - Validate prereg and source revision-plan boundary.

def read_json(path: Path):
    return json.loads(path.read_text())

def sha256(path: Path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

revision_summary = read_json(FORMULA_REVISION_PLAN_RUN / 'phase1_final_controls_metric_formula_revision_plan_summary.json')
revision_claim_state = read_json(FORMULA_REVISION_PLAN_RUN / 'phase1_final_controls_metric_formula_revision_claim_state.json')
revision_options = read_json(FORMULA_REVISION_PLAN_RUN / 'phase1_final_controls_metric_formula_options.json')

assert revision_summary.get('status') == 'phase1_final_controls_metric_formula_revision_plan_recorded'
assert revision_summary.get('manual_decision_required') is True
assert revision_summary.get('selected_formula') is None
assert revision_summary.get('code_change_allowed_now') is False
assert revision_summary.get('rerun_controls_allowed_now') is False
assert revision_summary.get('threshold_change_allowed_now') is False
assert revision_claim_state.get('headline_phase1_claim_open') is False

print(json.dumps({
    'formula_revision_plan_run': str(FORMULA_REVISION_PLAN_RUN),
    'controls_in_scope': revision_summary.get('controls_in_scope'),
    'runtime_formula_ids': revision_summary.get('runtime_formula_ids'),
    'candidate_formula_ids': [item.get('formula_id') for item in revision_options.get('candidate_formulas', [])],
    'claims_opened': revision_summary.get('claims_opened'),
}, indent=2))


In [ ]:
# Cell 4 - Record manual-hold plan and command preview.

PLAN_ROOT.mkdir(parents=True, exist_ok=True)
plan_dir = PLAN_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir.mkdir(parents=True, exist_ok=False)

runner_available = (REPO_DIR / 'src/phase1/final_controls_metric_formula_decision.py').exists()
preflight_blockers = []
if not runner_available:
    preflight_blockers.append('src/phase1/final_controls_metric_formula_decision.py missing')
if FORMULA_DECISION not in (None, 'raw_ba_ratio', 'gain_over_chance_ratio', 'unresolved'):
    preflight_blockers.append('FORMULA_DECISION must be None, raw_ba_ratio, gain_over_chance_ratio, or unresolved')

cmd = None
if FORMULA_DECISION is not None:
    cmd = [
        'python', '-m', 'src.cli', 'phase1_final_controls_metric_formula_decision',
        '--profile', 't4_safe',
        '--config', str(PREREG_BUNDLE),
        '--formula-revision-plan-run', str(FORMULA_REVISION_PLAN_RUN),
        '--formula-decision', FORMULA_DECISION,
        '--decision-rationale', DECISION_RATIONALE,
        '--output-root', str(OUTPUT_ROOT),
    ]

plan = {
    'status': 'phase1_final_controls_metric_formula_decision_manual_hold',
    'created_utc': plan_dir.name,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_CONTROLS_METRIC_FORMULA_DECISION,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'formula_revision_plan_run': str(FORMULA_REVISION_PLAN_RUN),
    'formula_decision': FORMULA_DECISION,
    'decision_rationale_present': bool(DECISION_RATIONALE.strip()),
    'preflight_blockers': preflight_blockers,
    'command': cmd,
    'scientific_integrity_rule': 'This notebook records a manual metric-formula decision only; it must not rerun controls or open claims.',
}
(plan_dir / 'phase1_final_controls_metric_formula_decision_manual_hold.json').write_text(json.dumps(plan, indent=2) + '\n')
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual launch gate.

if preflight_blockers:
    raise RuntimeError(f'Preflight blockers must be resolved before formula decision: {preflight_blockers}')
elif not RUN_FINAL_CONTROLS_METRIC_FORMULA_DECISION:
    print('HELD: Formula decision was not launched because manual flag is False.')
    print('NEXT: review the plan and decision memo, then rerun only with explicit claim-closed acknowledgement if appropriate.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not record formula decision without explicit claim-closed acknowledgement.')
elif FORMULA_DECISION is None:
    raise RuntimeError('FORMULA_DECISION is None. Choose raw_ba_ratio, gain_over_chance_ratio, or unresolved after review.')
elif not DECISION_RATIONALE.strip():
    raise RuntimeError('DECISION_RATIONALE is required and must explain the scientific/governance basis for the decision.')
else:
    run(cmd, cwd=REPO_DIR)
    print('Metric formula decision command completed. Review generated artifacts before any code/config change or rerun.')


In [ ]:
# Cell 6 - Inspect latest output if the decision was launched.

latest_output = None
if RUN_FINAL_CONTROLS_METRIC_FORMULA_DECISION and MANUAL_LAUNCH_ACK == REQUIRED_ACK and FORMULA_DECISION is not None:
    latest_output = resolve_latest(OUTPUT_ROOT, 'phase1_final_controls_metric_formula_decision_summary.json')
    summary = read_json(latest_output / 'phase1_final_controls_metric_formula_decision_summary.json')
    decision_record = read_json(latest_output / 'phase1_final_controls_metric_formula_decision_record.json')
    boundary = read_json(latest_output / 'phase1_final_controls_metric_formula_implementation_boundary.json')
    claim_state = read_json(latest_output / 'phase1_final_controls_metric_formula_decision_claim_state.json')

    print('Latest formula decision output:', latest_output)
    print('Formula decision:', summary.get('formula_decision'))
    print('Selected formula:', summary.get('selected_formula'))
    print('Controls in scope:', summary.get('controls_in_scope'))
    print('Code/config revision required:', summary.get('code_config_revision_required'))
    print('Code change allowed by this runner:', summary.get('code_change_allowed_by_this_runner'))
    print('Controls rerun by this runner:', summary.get('controls_rerun_by_decision_runner'))
    print('Controls rerun allowed by this runner:', summary.get('controls_rerun_allowed_by_this_runner'))
    print('Thresholds changed:', summary.get('thresholds_changed'))
    print('Claims opened:', summary.get('claims_opened'))
    print('Next step:', summary.get('next_step'))
    print('Claim blockers:', claim_state.get('blockers'))
else:
    print('No decision output inspected because launch is held.')


In [ ]:
# Cell 7 - Assertions and review note.

if latest_output is not None:
    assert summary.get('status') == 'phase1_final_controls_metric_formula_decision_recorded', summary.get('claim_blockers')
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('claims_opened') is False
    assert summary.get('existing_artifacts_reclassified') is False
    assert summary.get('thresholds_changed') is False
    assert summary.get('logits_or_metrics_edited') is False
    assert summary.get('controls_rerun_by_decision_runner') is False
    assert summary.get('code_change_allowed_by_this_runner') is False
    assert summary.get('controls_rerun_allowed_by_this_runner') is False
    assert decision_record.get('decision_is_claim_closed') is True
    assert claim_state.get('headline_phase1_claim_open') is False

    review_note = {
        'status': 'phase1_final_controls_metric_formula_decision_review_pass_claim_closed',
        'reviewed_run': str(latest_output),
        'formula_decision': summary.get('formula_decision'),
        'selected_formula': summary.get('selected_formula'),
        'checks_passed': [
            'decision_recorded',
            'claim_ready_false',
            'headline_claim_open_false',
            'existing_artifacts_not_reclassified',
            'thresholds_not_changed',
            'logits_or_metrics_not_edited',
            'controls_not_rerun_by_decision_runner',
            'code_change_not_performed_by_decision_runner',
        ],
        'next_step': summary.get('next_step'),
        'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    }
    note_path = latest_output / 'phase1_final_controls_metric_formula_decision_review_note.json'
    note_path.write_text(json.dumps(review_note, indent=2) + '\n')
    print('Review note written:', note_path)
    print(json.dumps(review_note, indent=2))
else:
    print('Assertions skipped because launch is held.')


In [ ]:
# Cell 8 - Closeout message.

print('================ Phase 1 Final Controls Metric Formula Decision Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_CONTROLS_METRIC_FORMULA_DECISION)
print('Formula revision-plan run:', FORMULA_REVISION_PLAN_RUN)
print('Formula decision:', FORMULA_DECISION)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)

if latest_output is None:
    print('HELD: Formula decision was not launched because manual flag is False or no formula decision was selected.')
    print('NEXT: choose raw_ba_ratio, gain_over_chance_ratio, or unresolved only after reviewing the revision-plan decision memo.')
else:
    print('Latest formula decision output:', latest_output)
    print('Selected formula:', summary.get('selected_formula'))
    print('Code/config revision required:', summary.get('code_config_revision_required'))
    print('Code change allowed by this runner:', summary.get('code_change_allowed_by_this_runner'))
    print('Controls rerun allowed by this runner:', summary.get('controls_rerun_allowed_by_this_runner'))
    print('Next step:', summary.get('next_step'))
    print('CHECK REQUIRED: Review phase1_final_controls_metric_formula_decision_memo.md before any code/config remediation or control rerun.')

print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
